# Seguridad del Data Lake

SAIF aplicado al Data Lake real en MinIO (bucket `datalake`). La verificacion de
integridad vive en `tp_mlops2.datalake.cargar_verificando`, y es la misma funcion que usa
`load_model_from_lake()` -la que las APIs usarian si cargaran el modelo directo del lake
en vez de via MLflow. `scripts/upload_to_datalake.py` publica el checksum SHA-256 junto
al modelo cada vez que sube una version nueva.

In [1]:
from tp_mlops2.datalake import (
    BUCKET, MODEL_VERSION_PREFIX, cargar_verificando, get_s3_client, load_model_from_lake,
)

s3 = get_s3_client()
key_modelo = f"{MODEL_VERSION_PREFIX}/model.joblib"
key_checksum = f"{MODEL_VERSION_PREFIX}/model.sha256"

objetos = s3.list_objects_v2(Bucket=BUCKET, Prefix=f"{MODEL_VERSION_PREFIX}/")["Contents"]
for o in objetos:
    print(o["Key"], f"{o['Size']:,} bytes")

models/v1/feature_cols.joblib 202 bytes
models/v1/model.joblib 32,625,393 bytes
models/v1/model.sha256 64 bytes


## 1. Carga legitima: el checksum coincide

`load_model_from_lake()` descarga el modelo real y, antes de dejarlo usar, recalcula su
SHA-256 y lo compara contra `model.sha256`, publicado por el propio
`upload_to_datalake.py` al momento de subir el modelo.

In [2]:
model, feature_cols = load_model_from_lake()
print(f"cargado y verificado: {len(feature_cols)} features")

cargado y verificado: 14 features


## 2. Manipulacion del bucket: detectada

Simulamos que alguien reemplaza el artefacto del modelo sin actualizar su checksum -
exactamente lo que pasaria si un atacante, o un error de despliegue, corrompiera el
archivo real en producción.

In [3]:
original = s3.get_object(Bucket=BUCKET, Key=key_modelo)["Body"].read()

s3.put_object(Bucket=BUCKET, Key=key_modelo, Body=b"modelo-manipulado-por-un-atacante")

try:
    cargar_verificando(s3, BUCKET, key_modelo, key_checksum)
    print("ERROR: no deberia haber cargado")
except ValueError as e:
    print("detectado:", e)

detectado: INTEGRIDAD ROTA: esperado e88cdc19cafeecb1..., obtenido 4e4da7851f3c308f... - el artefacto fue manipulado, no se carga.


## 3. Restauracion del modelo real

El bucket real del proyecto no puede quedar roto despues de esta prueba.

In [4]:
s3.put_object(Bucket=BUCKET, Key=key_modelo, Body=original)

model_restaurado, _ = load_model_from_lake()
print("modelo real restaurado y verificado de nuevo")

modelo real restaurado y verificado de nuevo


## Reflexion

`cargar_verificando` no es una funcion de ejemplo: es la misma que usa
`load_model_from_lake()`, la ruta directa al Data Lake que corre independiente de MLflow.
Un modelo manipulado en el bucket -por un atacante o por un error de despliegue- nunca se
carga, porque el checksum publicado junto al artefacto real es la referencia contra la que
se verifica cada descarga.